# Bacon–Shor memory: dedicated SE and MWPM

**[DEMO]** — uses the packaged `BaconShorCode`, `BaconShorCodeExtractionBlock`,
`MemoryExperiment` and `SimulationPipeline`. Detectors and the protected logical
observable are generated automatically by LightStim.

This is the square weight-2 subsystem code. The patch declares its stabilizer
centre and gauge generators once; the tracker follows the alternating gauge
measurements. No manual active-stabilizer updates are needed.


In [1]:
import sys, json
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "lightstim").is_dir() and (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.ir.qec_system import QECSystem
from lightstim.noise.config import NoiseConfig
from lightstim.protocols.memory import MemoryExperiment
from lightstim.qec_code.bacon_shor import BaconShorCode, BaconShorCodeExtractionBlock
from lightstim.simulation.decoder_backend import DecoderConfig, SimulationPipeline
# Experiment-specific selection of existing detectors; no new detectors are constructed.
from benchmarks.memory.bacon_shor.run import select_memory_detectors, plot_schedule
import pandas as pd
from IPython.display import display

## Patch and physical layout

The square family has $[[n,k,r_g,d]]=[[d^2,1,(d-1)^2,d]]$.
Horizontal nearest-neighbor XX and vertical nearest-neighbor ZZ operators form
$2d(d-1)$ gauge generators. The stabilizer centre has $2(d-1)$ generators,
formed by products across whole strips; they have no separate readout ancillas.

Data qubits are at $(2c,2r)$, X-gauge ancillas at $(2c+1,2r)$ and Z-gauge
ancillas at $(2c,2r+1)$. At d=3 this gives 9 data + 12 readout ancillas.
The four gauge **qubits** are subsystem degrees of freedom, not these ancillas.


In [2]:
patch = BaconShorCode(distance=3)
system = QECSystem()
system.add_patch(patch, name="bs")
se = BaconShorCodeExtractionBlock(system)
print("data / readout ancillas:", len(patch.data_indices), len(patch.syndrome_indices))
print("centre / gauge generators:", len(patch.stabilizers), len(patch.gauges))
print("CNOT layers: X =", se.depth_x, "; Z =", se.depth_z, "; XZ pair =", se.cnot_depth)
assert se.cnot_depth == 4

data / readout ancillas: 9 12
centre / gauge generators: 4 12
CNOT layers: X = 2 ; Z = 2 ; XZ pair = 4


## Dedicated syndrome extraction

Each X gauge is measured with ancilla → left data, then ancilla → right data.
Each Z gauge uses upper data → ancilla, then lower data → ancilla in this
row-index drawing. X extraction precedes Z extraction. Each basis has its own
reset and terminal measurement block.

The schedule has **four CNOT layers per complete XZ round**, independent of d;
reset and measurement time are additional. The figure uses the actual pairs
in `se.x_layers` and `se.z_layers`, with the patch's physical qubit indices.

![Dedicated Bacon–Shor CNOT schedule](assets/bacon_shor/dedicated_se.png)

To redraw after changing the patch, use `plot_schedule(patch, se)` for d=3.
The generic CSS edge-colored implementation remains an explicit alternative.


In [3]:
# Actual dedicated SE instructions, before noise and detector annotation.
print(se.circuit)

RX 9 10 11 12 13 14
TICK[SE_start]
CX 9 0 10 1 11 3 12 4 13 6 14 7
TICK
CX 9 1 10 2 11 4 12 5 13 7 14 8
TICK
MX 9 10 11 12 13 14
TICK
R 15 16 17 18 19 20
TICK[SE_start]
CX 0 15 1 16 2 17 3 18 4 19 5 20
TICK
CX 3 15 4 16 5 17 6 18 7 19 8 20
TICK
M 15 16 17 18 19 20


## Two-round memory circuit

Use the same dedicated block inside the normal memory protocol. The small
visualization includes initialization, two full XZ rounds, final data readout
and automatically inferred detectors. The independent X-memory check below
also verifies the protected logical observable.


In [4]:
memory = MemoryExperiment(
    qec_system=system,
    extraction_block_class=BaconShorCodeExtractionBlock,
    rounds=2, basis="Z",
)
circuit = memory.build()
x_circuit = MemoryExperiment(qec_patch=BaconShorCode(distance=3), rounds=2, basis="X").build()
assert not circuit.compile_detector_sampler(seed=71).sample(256, append_observables=True).any()
assert not x_circuit.compile_detector_sampler(seed=71).sample(256, append_observables=True).any()
print("physical qubits / detectors / logical observables:",
      circuit.num_qubits, circuit.num_detectors, circuit.num_observables)
# Optional interactive redraw; keep large SVG output cleared when committing:
# circuit.diagram("detslice-with-ops-svg")

physical qubits / detectors / logical observables: 21 12 1


![Two-round Bacon–Shor Z memory with automatic detectors](assets/bacon_shor/memory_d3.svg)

[Open the full SVG](assets/bacon_shor/memory_d3.svg) to inspect the circuit.


## CPU MWPM decoding

Use native `circuit_level` noise with `p_2q = p_reset = p_meas = 0.001` and
`p_idle = p_1q = 0`: CNOT depolarization, reset flips and measurement flips,
**including final data readout**. There is no idle noise.

Build the full X/Z detector set first, then select its original pure Z-record
detectors for Z memory. All physical gates, noise, measurements and the logical
observable are preserved. The selected, undecomposed DEM is graphlike for
this experiment. The full X/Z DEM contains hyperedges; locality by itself
is not a reason to feed it directly to MWPM. Selecting one basis drops
complementary syndrome information, so this is a baseline decoder.

This cell is a small live decoding check; the next section shows the saved
1M-shot results. `rounds=d` means d complete XZ pairs per memory shot.


In [5]:
p = 0.001
noise = NoiseConfig(p_2q=p, p_reset=p, p_meas=p)
full = MemoryExperiment(
    qec_patch=BaconShorCode(distance=3),
    extraction_block_class=BaconShorCodeExtractionBlock,
    basis="Z", rounds=3, noise_params=noise, noise_model="circuit_level",
).build()
mwpm_circuit = select_memory_detectors(full, "Z")
dem = mwpm_circuit.detector_error_model(decompose_errors=False)
assert all(sum(t.is_relative_detector_id() for t in op.targets_copy()) <= 2
           for op in dem.flattened() if op.type == "error")
print("full / MWPM detectors:", full.num_detectors, mwpm_circuit.num_detectors)
stats = SimulationPipeline(
    DecoderConfig("pymatching", backend="cpu"),
    max_shots=10_000, max_errors=10_001, num_workers=1,
    batch_size=10_000, print_progress=False,
).run(mwpm_circuit)
print(f"Live smoke check: {stats.errors}/{stats.shots}; LER per shot = {stats.logical_error_rate:.3g}")

full / MWPM detectors: 16 12
Live smoke check: 10/10000; LER per shot = 0.001


## Saved MWPM results

**2026-09-07 native-noise baseline:** same dedicated SE and noise parameters
as above; 1,000,000 fixed shots per distance, no postselection. The metric is
logical error probability for the **complete d-round memory shot**, not per
round. Error bars are pointwise 95% exact Clopper–Pearson intervals.


In [6]:
summary_path = ROOT / "benchmarks/memory/bacon_shor/precompute/summary.json"
summary = json.loads(summary_path.read_text())
rows = pd.DataFrame(summary["results"])
display(rows[["d", "rounds", "p", "errors", "shots", "ler", "ci95"]])

,d,rounds,p,errors,shots,ler,ci95
0,3,3,0.001,708,1000000,0.000708,"[0.0006568187561224911, 0.0007621080374451837]"
1,5,5,0.001,201,1000000,0.000201,"[0.00017417389481745092, 0.0002307876122702327]"
2,7,7,0.001,102,1000000,0.000102,"[8.316955171703957e-05, 0.00012381968963397926]"
3,9,9,0.001,63,1000000,0.000063,"[4.841125873208777e-05, 8.060365778585897e-05]"


![Dedicated Bacon–Shor CPU MWPM results](assets/bacon_shor/mwpm.png)

The LER decreases over d=3,5,7,9 at this p. These finite-size results do not
establish an asymptotic threshold. For these four circuits and this fault set,
the benchmark also obtains matching circuit-distance lower and upper bounds d:
the complete selected graphlike DEM supplies the lower bound and a physical
undetected logical-fault witness in the full DEM supplies the upper bound.

The summary records source/circuit hashes, seeds, shot counts and package
versions. Reproduce the large run outside the notebook:

```bash
python -m benchmarks.memory.bacon_shor.run --shots 1000000
```

- [Benchmark configuration and provenance](../../benchmarks/memory/bacon_shor/README.md)
- [Code and dedicated SE implementation](../../lightstim/qec_code/bacon_shor/README.md)
- [Historical reviews in playground](../../playground/subsystem/README.md): generic/dedicated
  comparisons, literature checks and paired CPU BP+OSD/MWPM results. Their earlier
  circuit-noise baseline used **ideal final data readout**, so its counts should
  not be mixed with this notebook's noisy-final-readout baseline.
- [SHYPS memory and visualization](memory_subsystem.ipynb)
